# Stochastic volatility on the five largest cryptocurrencies with oWALNUTS

This notebook samples the **full posterior** of a standard stochastic-volatility model —
$r_t = e^{h_t/2}\varepsilon_t$, $h_t = \mu + \phi(h_{t-1}-\mu) + \sigma\eta_t$ —
over every daily close in each asset's history (T up to 3,153, so up to 3,156 parameters),
using [oWALNUTS](https://github.com/dhruv575/oWALNUTS): a Rust implementation of the
within-orbit adaptive leapfrog No-U-Turn sampler (JMLR 2026) with linear-time structured
metrics and a one-line PyMC entry point.

Everything here is backed by the preregistered study
`STUDIES/flagship_crypto_sv_v1` in the oWALNUTS repository (protocol, seeds, checksums,
and a ledger entry); numbers shown below are loaded from its artifacts.

## Why care: NUTS can silently miss part of the posterior

On Neal's funnel — the classic hard geometry, with an exactly known marginal —
NumPyro's NUTS places **almost zero** mass below $\omega=-5$ where the true mass is 4.78%,
emitting thousands of divergences; oWALNUTS at the paper's tuning matches the analytic
marginal on every seed with zero divergences
(study `STUDIES/numpyro_comparisons_v10_v1`, kernel v10):

![funnel](crypto_sv_assets/funnel.png)

In [1]:
import json, pathlib, numpy as np
assets = pathlib.Path('crypto_sv_assets')
doc = json.loads((assets / 'BTC.json').read_text())
closes = np.array([row[1] for row in doc['closes']])
r = np.log(closes[1:] / closes[:-1])
print(f"BTC-USDT daily closes {doc['first']} to {doc['last']}  (T = {len(r)} returns)")

BTC-USDT daily closes 2018-01-11 to 2026-08-30  (T = 3153 returns)


## One line from PyMC

Define the SV model in PyMC as usual; `owalnuts.from_pymc(model, gil_free=True)` compiles
the joint log-density gradient to a GIL-free callback, so four chains sample in parallel
from Rust. The structured metric — the AR(1)-plus-curvature tridiagonal precision of the
latent path — is one call. *(Demo budget below: 300 warmup / 500 draws so the cell runs in
seconds; the study runs 1,000/3,000 — see the results table.)*

In [2]:
import owalnuts, pymc as pm, pytensor.tensor as pt
T = len(r)
with pm.Model() as model:
    mu = pm.Normal('mu', -10, 5)
    phi_raw = pm.Beta('phi_raw', 20, 1.5)
    phi = 2 * phi_raw - 1
    sigma = pm.HalfNormal('sigma', 0.5)
    pm.AR('h', rho=[mu * (1 - phi), phi], sigma=sigma, constant=True,
          init_dist=pm.Normal.dist(mu, sigma / pt.sqrt(1 - phi**2)), shape=T)
    pm.Normal('r', 0.0, pt.exp(model['h'] / 2), observed=r)

target, dim, q0, names, unravel = owalnuts.from_pymc(model, gil_free=True)

# One-shot structured metric from the study's stage-A calibration (see PREREGISTRATION.md)
cal = json.loads((assets / 'BTC-calibration-97001.json').read_text())
phi_h, s2 = cal['phi_hat'], cal['sigma_hat'] ** 2
h_mean = np.asarray(cal['h_mean'])
diag = np.full(T, (1 + phi_h**2) / s2); diag[[0, -1]] = 1 / s2
diag += 0.5 * r**2 * np.exp(-h_mean)
off = np.full(T - 1, -phi_h / s2)
cov = 2.0 * np.asarray(cal['global_cov']).reshape(3, 3)
prec = np.linalg.inv(cov)
mass = owalnuts.tridiagonal_precision_mass(
    np.array([prec[0, 0], prec[1, 1], prec[2, 2]]), np.array([prec[0, 1], prec[1, 2]])
) + owalnuts.tridiagonal_precision_mass(diag, off)

base = np.concatenate([[cal['mu_hat'], cal['a_hat'], cal['s_hat']], h_mean])
starts = base + 0.3 * np.random.default_rng(1).standard_normal((4, dim))
result = owalnuts.sample(
    target, dim, init=starts, chains=4, warmup=300, draws=500, seed=1, threads=4,
    tuning=owalnuts.Tuning(step_size=0.1, max_depth=9, max_refinement_levels=6),
    adaptation=owalnuts.Adaptation(adapt_mass=False, paper=owalnuts.PaperAdaptation()),
    mass=mass, max_target_evaluations=1_000_000_000)
print(f'wall {result.wall_seconds:.1f}s, {result.target_calls:,} gradient evaluations, '
      f'divergences {int(result.divergent.sum())}')
post = result.samples
phi_draws = 2 / (1 + np.exp(-post[:, :, 1])) - 1
print(f'phi = {phi_draws.mean():.3f}, sigma = {np.exp(post[:, :, 2]).mean():.3f} (demo budget)')

C:\dev\oWALNUTS\integrations\python\.venv\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


wall 3.5s, 94,211 gradient evaluations, divergences 0
phi = 0.810, sigma = 0.683 (demo budget)


## The five-asset study (preregistered evidence)


In [3]:
import pandas as pd
summary = json.loads((assets / 'summary.json').read_text())
rows = []
for res in summary['results']:
    if res.get('missing'):
        continue
    d, m = res['diagnostics'], res['meta']
    rows.append({
        'asset': res['symbol'], 'cell': res['cell'], 'seed': res['seed'],
        'primary health': 'PASS' if res['gates']['primary'] else 'fail',
        'globals gate': 'PASS' if res['gates']['globals'] else 'fail',
        'min primary ESS': round(res['min_bulk_primary']),
        'phi': round(d['phi_mean'], 3), 'sigma': round(d['sigma_mean'], 3),
        'wall s': round(m['wall_sampling'], 1), 'divergences': m['divergences'],
    })
pd.DataFrame(rows)

,asset,cell,seed,primary health,globals gate,min primary ESS,phi,sigma,wall s,divergences
0,BTC,native,97001,fail,fail,171,0.812,0.680,17.6,0
1,BTC,native,97002,fail,fail,193,0.812,0.680,21.1,0
2,BTC,native,97003,fail,fail,227,0.812,0.682,17.3,0
3,BTC,pymc,97001,fail,fail,218,0.809,0.687,19.4,0
4,BTC,pymc,97002,fail,fail,274,0.807,0.691,19.0,0
5,BTC,pymc,97003,fail,fail,187,0.811,0.683,21.3,0
6,BTC,nutpie,97001,fail,PASS,369,0.812,0.680,25.5,0
7,BTC,numpyro,97001,PASS,PASS,420,0.814,0.675,46.9,0
8,ETH,native,97001,fail,PASS,386,0.839,0.588,21.3,0
9,ETH,native,97002,fail,fail,398,0.841,0.585,18.6,0


## Posterior volatility paths

Annualized volatility $100\sqrt{365}\,e^{h_t/2}$ with 90% posterior bands, from the
oWALNUTS evidence runs:

![volatility](crypto_sv_assets/volatility.png)

## Honest limitations

- **Not every cell passed the preregistered health gates.** At this budget oWALNUTS passes
  primary health on XRP/SOL/BNB but not BTC/ETH (T ≈ 3,150), where NumPyro/nutpie's
  windowed adaptation extracts more global ESS; and the `from_pymc` arm hit stuck seeds on
  ETH/SOL (R-hat up to 1.65, zero divergences) from unlucky starts under the frozen metric.
  Full tables: `STUDIES/flagship_crypto_sv_v1/artifacts/RESULTS.md`.
- **The global ridge is everyone's bottleneck.** corr$(a,s)\approx-0.9$ throttles the
  effective sample size of $(\phi, \sigma)$ in *every* backend tested (oWALNUTS, nutpie,
  NumPyro). Block-diagonal metrics cannot express the global-path coupling; that is the
  next research item (the arrowhead line).
- **Pure-JAX models remain NumPyro's home turf** — oWALNUTS' advantage needs a compiled
  gradient (PyMC/numba, Stan via BridgeStan, or Rust).
- Wall times were measured on a shared machine; ESS per gradient evaluation is the robust
  comparison and is in the study artifacts.
- The $\sigma_x \to 0$ funnel boundary of state-space models (fixture sspd-10) is unsolved
  by every Euclidean sampler tested, including NUTS at depth 12.
